# 05 Embeddings and Vector Store

## Goal

This notebook turns RAG-ready SEC chunks into searchable vector embeddings.

The workflow is:

```text
RAG chunks
→ embedding model
→ vector embeddings
→ Chroma vector database
→ similarity search
→ baseline retrieval results
```

This is the first notebook where the project becomes a working retrieval system.

At the end, we should be able to ask a question like:

```text
What are NVIDIA's AI-related risks?
```

And retrieve relevant SEC filing chunks with citation metadata.

In [6]:
# Import tools for working with files and folders
from pathlib import Path

# Import pandas for table operations
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import ChromaDB for vector database storage and retrieval
import chromadb

# Import SentenceTransformer for local embedding generation
from sentence_transformers import SentenceTransformer

# Import time to measure embedding and retrieval speed
import time

In [7]:
# Detect the project root automatically
# If this notebook is inside the notebooks folder, move one level up
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Create main project paths
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
VECTORSTORE_DIR = DATA_DIR / "vectorstore"

# Create Chroma vector database folder
CHROMA_DIR = VECTORSTORE_DIR / "chroma_sec_10k"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# Print paths to confirm everything is correct
print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)
print("Vectorstore folder:", VECTORSTORE_DIR)
print("Chroma folder:", CHROMA_DIR)

Project root: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI
Processed data folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed
Vectorstore folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore
Chroma folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore\chroma_sec_10k


In [8]:
# Set path to the final RAG chunks from notebook 04
rag_chunks_file = PROCESSED_DIR / "sec_10k_rag_chunks.csv"

# Check that the chunk file exists before loading it
if not rag_chunks_file.exists():
    raise FileNotFoundError(
        f"Could not find {rag_chunks_file}. Run 04_chunking_experiments.ipynb first."
    )

# Load the RAG chunk dataset
rag_chunks_df = pd.read_csv(rag_chunks_file)

# Preview the chunks
rag_chunks_df.head()

,chunk_id,document_id,ticker,company_name,filing_date,accession_number,filing_url,section_name,source_label,citation_label,chunk_index,chunk_size,chunk_overlap,start_word,end_word,chunk_word_count,chunk_character_count,chunk_text
0,AAPL_2025-10-31_item_1_business_chunk_0000,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 0",0,350,75,0,350,350,2125,Item 1. Business Company Background The Compan...
1,AAPL_2025-10-31_item_1_business_chunk_0001,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 1",1,350,75,275,625,350,2341,Apple Inc. | 2025 Form 10-K | 1 Services Adver...
2,AAPL_2025-10-31_item_1_business_chunk_0002,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 2",2,350,75,550,900,350,2429,"Greater China includes China mainland, Hong Ko..."
3,AAPL_2025-10-31_item_1_business_chunk_0003,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 3",3,350,75,825,1175,350,2563,by imitating the Company’s products and infrin...
4,AAPL_2025-10-31_item_1_business_chunk_0004,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 4",4,350,75,1100,1450,350,2357,provide products and services at little or no ...


In [9]:
# Print the number of rows and columns
print("Rows and columns:", rag_chunks_df.shape)

# Print number of unique companies
print("Companies:", rag_chunks_df["ticker"].nunique())

# Print number of unique section types
print("Section types:", rag_chunks_df["section_name"].nunique())

# Print total chunks
print("Total chunks:", len(rag_chunks_df))

# Display chunk count by company
rag_chunks_df["ticker"].value_counts()

Rows and columns: (520, 18)
Companies: 5
Section types: 4
Total chunks: 520


ticker
AMD     214
NVDA    111
TSLA     74
MSFT     64
AAPL     57
Name: count, dtype: int64

In [10]:
# Define columns required for embedding and retrieval
required_columns = [
    "chunk_id",
    "ticker",
    "company_name",
    "filing_date",
    "accession_number",
    "filing_url",
    "section_name",
    "source_label",
    "citation_label",
    "chunk_text"
]

# Find missing columns
missing_columns = [
    column for column in required_columns
    if column not in rag_chunks_df.columns
]

# Stop notebook if required columns are missing
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Check for missing chunk text
missing_text_count = rag_chunks_df["chunk_text"].isna().sum()

# Check for duplicate chunk IDs
duplicate_id_count = rag_chunks_df["chunk_id"].duplicated().sum()

# Print validation results
print("Missing chunk text:", missing_text_count)
print("Duplicate chunk IDs:", duplicate_id_count)

# Stop if chunk text is missing
if missing_text_count > 0:
    raise ValueError("Some chunks have missing text.")

# Stop if chunk IDs are duplicated
if duplicate_id_count > 0:
    raise ValueError("Some chunk IDs are duplicated.")

print("Chunk dataset is valid.")

Missing chunk text: 0
Duplicate chunk IDs: 0
Chunk dataset is valid.


In [11]:
rag_chunks_df["chunk_text"]

0      Item 1. Business Company Background The Compan...
1      Apple Inc. | 2025 Form 10-K | 1 Services Adver...
2      Greater China includes China mainland, Hong Ko...
3      by imitating the Company’s products and infrin...
4      provide products and services at little or no ...
                             ...                        
515    loans with pledges of Tesla common stock that ...
516    Item 7, Management's Discussion and Analysis o...
517    billion of unused committed credit amounts as ...
518    net income excluding non-cash expenses, gains ...
519    ITEM 7A. QUANTITATIVE AND QUALITATIVE DISCLOSU...
Name: chunk_text, Length: 520, dtype: object

## Embedding Model

For the first version, we will use:

```text
sentence-transformers/all-MiniLM-L6-v2
```

Why this model:

- Free
- Runs locally
- Fast
- Good enough for semantic search
- Easy to explain in interviews

Later, we can upgrade to stronger embedding models such as BGE, E5, OpenAI embeddings, or domain-specific finance embeddings.

In [12]:
# Set the embedding model name
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Start timer
start_time = time.time()

# Load the SentenceTransformer embedding model
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# End timer
end_time = time.time()

# Print model loading time
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Model loaded in seconds:", round(end_time - start_time, 2))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\.venv\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tevin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Model loaded in seconds: 6.58


#### Test embedding on one chunk

In [13]:
# Select one sample chunk
sample_chunk_text = rag_chunks_df.iloc[0]["chunk_text"]

# Create embedding for the sample chunk
sample_embedding = embedding_model.encode(
    sample_chunk_text,
    normalize_embeddings=True
)

# Print embedding information
print("Embedding type:", type(sample_embedding))
print("Embedding shape:", sample_embedding.shape)
print("First 10 values:", sample_embedding[:10])

Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)
First 10 values: [-0.08125816 -0.00184671  0.04513849 -0.08958755  0.03296055 -0.01161384
  0.04858047  0.01989671  0.06138504 -0.01719774]


#### Prepare text and metadata

In [17]:
# Create a clean copy of the chunk dataset
chunks_for_vectorstore = rag_chunks_df.copy()

# Make sure chunk IDs are strings
chunks_for_vectorstore["chunk_id"] = chunks_for_vectorstore["chunk_id"].astype(str)

# Make sure chunk text is clean text
chunks_for_vectorstore["chunk_text"] = (
    chunks_for_vectorstore["chunk_text"]
    .astype(str)
    .str.strip()
)

# Fill missing filing URLs with an empty string
chunks_for_vectorstore["filing_url"] = chunks_for_vectorstore["filing_url"].fillna("")

# Fill any other missing values with empty strings
chunks_for_vectorstore = chunks_for_vectorstore.fillna("")

# Preview vectorstore-ready data
chunks_for_vectorstore[
    [
        "chunk_id",
        "ticker",
        "section_name",
        "citation_label",
        "chunk_text"
    ]
].head()

,chunk_id,ticker,section_name,citation_label,chunk_text
0,AAPL_2025-10-31_item_1_business_chunk_0000,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 0",Item 1. Business Company Background The Compan...
1,AAPL_2025-10-31_item_1_business_chunk_0001,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 1",Apple Inc. | 2025 Form 10-K | 1 Services Adver...
2,AAPL_2025-10-31_item_1_business_chunk_0002,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 2","Greater China includes China mainland, Hong Ko..."
3,AAPL_2025-10-31_item_1_business_chunk_0003,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 3",by imitating the Company’s products and infrin...
4,AAPL_2025-10-31_item_1_business_chunk_0004,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 4",provide products and services at little or no ...


#### Create embeddings for all chunks

In [23]:
# Extract chunk texts as a list
chunk_texts = chunks_for_vectorstore["chunk_text"].tolist()

# Start timer
start_time = time.time()

# Create embeddings for all chunks
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

# End timer
end_time = time.time()

# Print embedding summary
print("Created embeddings for chunks:", len(chunk_embeddings))
print("Embedding matrix shape:", chunk_embeddings.shape)
print("Embedding time in seconds:", round(end_time - start_time, 2))

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Created embeddings for chunks: 520
Embedding matrix shape: (520, 384)
Embedding time in seconds: 21.48


#### Build Chroma Meta data

In [18]:
# Create metadata records for Chroma
# Chroma metadata values should be simple values like strings, ints, floats, or booleans
metadata_records = []

# Loop through each chunk row
for _, row in chunks_for_vectorstore.iterrows():

    # Store citation and filtering metadata
    metadata_records.append({
        "chunk_id": str(row["chunk_id"]),
        "document_id": str(row["document_id"]),
        "ticker": str(row["ticker"]),
        "company_name": str(row["company_name"]),
        "filing_date": str(row["filing_date"]),
        "accession_number": str(row["accession_number"]),
        "filing_url": str(row["filing_url"]),
        "section_name": str(row["section_name"]),
        "source_label": str(row["source_label"]),
        "citation_label": str(row["citation_label"]),
        "chunk_index": int(row["chunk_index"]),
        "chunk_word_count": int(row["chunk_word_count"])
    })

# Preview one metadata record
metadata_records[0]

{'chunk_id': 'AAPL_2025-10-31_item_1_business_chunk_0000',
 'document_id': 'AAPL_2025-10-31_item_1_business',
 'ticker': 'AAPL',
 'company_name': 'Apple',
 'filing_date': '2025-10-31',
 'accession_number': '0000320193-25-000079',
 'filing_url': 'https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm',
 'section_name': 'item_1_business',
 'source_label': 'AAPL 2025-10-31 10-K item_1_business',
 'citation_label': 'AAPL 2025-10-31 10-K, item_1_business, chunk 0',
 'chunk_index': 0,
 'chunk_word_count': 350}

#### Create Chroma persistent client

In [19]:
# Create a persistent Chroma client
# This saves the vector database locally inside data/vectorstore
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

# Define collection name
COLLECTION_NAME = "sec_10k_rag_chunks"

# Display Chroma setup
print("Chroma directory:", CHROMA_DIR)
print("Collection name:", COLLECTION_NAME)

Chroma directory: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore\chroma_sec_10k
Collection name: sec_10k_rag_chunks


#### Reset or create Chroma collection

In [20]:
# Set this to True while developing so rerunning the notebook starts clean
RESET_COLLECTION = True

# Delete existing collection if reset is enabled
if RESET_COLLECTION:
    try:
        chroma_client.delete_collection(name=COLLECTION_NAME)
        print("Deleted existing collection:", COLLECTION_NAME)
    except Exception:
        print("No existing collection to delete.")

# Create or get the Chroma collection
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "description": "SEC 10-K RAG chunks for RiskRadar AI",
        "embedding_model": EMBEDDING_MODEL_NAME
    }
)

# Confirm collection was created
print("Collection ready:", COLLECTION_NAME)

No existing collection to delete.
Collection ready: sec_10k_rag_chunks


#### Add chunks to Chroma

In [24]:
# Convert Chroma inputs to Python lists
ids = chunks_for_vectorstore["chunk_id"].astype(str).tolist()
documents = chunks_for_vectorstore["chunk_text"].tolist()
embeddings = chunk_embeddings.tolist()

# Set batch size for adding records to Chroma
BATCH_SIZE = 500

# Add records to Chroma in batches
for start_idx in range(0, len(ids), BATCH_SIZE):

    # Calculate batch end index
    end_idx = start_idx + BATCH_SIZE

    # Add this batch to Chroma
    collection.add(
        ids=ids[start_idx:end_idx],
        documents=documents[start_idx:end_idx],
        metadatas=metadata_records[start_idx:end_idx],
        embeddings=embeddings[start_idx:end_idx]
    )

    # Print progress
    print(f"Added records {start_idx} to {min(end_idx, len(ids))}")

# Print final collection count
print("Total records in Chroma collection:", collection.count())

Added records 0 to 500
Added records 500 to 520
Total records in Chroma collection: 520


#### Save embedding metadata summary

In [25]:
# Create an embedding summary table
embedding_summary = pd.DataFrame({
    "setting": [
        "embedding_model",
        "total_chunks",
        "embedding_dimensions",
        "vectorstore",
        "collection_name",
        "chroma_path"
    ],
    "value": [
        EMBEDDING_MODEL_NAME,
        len(rag_chunks_df),
        chunk_embeddings.shape[1],
        "ChromaDB",
        COLLECTION_NAME,
        str(CHROMA_DIR)
    ]
})

# Set output path
embedding_summary_file = PROCESSED_DIR / "sec_10k_embedding_summary.csv"

# Save embedding summary
embedding_summary.to_csv(embedding_summary_file, index=False)

# Display summary
embedding_summary

,setting,value
0,embedding_model,sentence-transformers/all-MiniLM-L6-v2
1,total_chunks,520
2,embedding_dimensions,384
3,vectorstore,ChromaDB
4,collection_name,sec_10k_rag_chunks
5,chroma_path,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...


## Baseline Retrieval

Now we test whether the vector database can retrieve relevant filing chunks.

The retrieval flow is:

```text
user question
→ embed the question
→ search Chroma
→ return closest chunks
→ inspect citations and evidence
```

This is not answer generation yet.

This is retrieval only.

In [31]:
def search_sec_chunks(query, top_k=5, ticker=None, section_name=None):
    """
    Search SEC filing chunks using semantic similarity.

    Optional filters:
    - ticker
    - section_name
    """

    # Embed the user query
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # Create a list to store Chroma filter conditions
    filter_conditions = []

    # Add ticker filter if provided
    if ticker is not None:
        filter_conditions.append({"ticker": ticker})

    # Add section filter if provided
    if section_name is not None:
        filter_conditions.append({"section_name": section_name})

    # If there are no filters, use None
    if len(filter_conditions) == 0:
        where_filter = None

    # If there is only one filter, pass it directly
    elif len(filter_conditions) == 1:
        where_filter = filter_conditions[0]

    # If there are multiple filters, use Chroma's $and operator
    else:
        where_filter = {"$and": filter_conditions}

    # Query Chroma collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )

    # Create an empty list for readable result records
    result_records = []

    # Loop through returned results
    for rank, (doc, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ),
        start=1
    ):

        # Store search result with citation metadata
        result_records.append({
            "rank": rank,
            "distance": distance,
            "ticker": metadata["ticker"],
            "company_name": metadata["company_name"],
            "filing_date": metadata["filing_date"],
            "section_name": metadata["section_name"],
            "citation_label": metadata["citation_label"],
            "filing_url": metadata["filing_url"],
            "chunk_text": doc
        })

    # Return results as a DataFrame
    return pd.DataFrame(result_records)

#### Test search: NVIDIA AI risks

In [32]:
# Define a test question
query = "What are NVIDIA's risks related to artificial intelligence and competition?"

# Search only NVIDIA chunks
search_results = search_sec_chunks(
    query=query,
    top_k=5,
    ticker="NVDA"
)

# Display search results
search_results[
    [
        "rank",
        "distance",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,rank,distance,ticker,section_name,citation_label
0,1,0.827991,NVDA,item_7_mda,"NVDA 2026-02-25 10-K, item_7_mda, chunk 0"
1,2,0.905708,NVDA,item_1_business,"NVDA 2026-02-25 10-K, item_1_business, chunk 1"
2,3,0.964376,NVDA,item_1_business,"NVDA 2026-02-25 10-K, item_1_business, chunk 4"
3,4,1.044540,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch..."
4,5,1.067148,NVDA,item_1_business,"NVDA 2026-02-25 10-K, item_1_business, chunk 7"


#### Inspect retrieved evidence

In [33]:
# Select the top search result
top_result = search_results.iloc[0]

# Print citation metadata
print("Rank:", top_result["rank"])
print("Ticker:", top_result["ticker"])
print("Company:", top_result["company_name"])
print("Section:", top_result["section_name"])
print("Citation:", top_result["citation_label"])
print("Filing URL:", top_result["filing_url"])
print("Distance:", top_result["distance"])

# Preview retrieved text
print(top_result["chunk_text"][:2500])

Rank: 1
Ticker: NVDA
Company: NVIDIA
Section: item_7_mda
Citation: NVDA 2026-02-25 10-K, item_7_mda, chunk 0
Filing URL: https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm
Distance: 0.8279906511306763
Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations The following discussion and analysis of our financial condition and results of operations should be read in conjunction with “Item 1A. Risk Factors,” our Consolidated Financial Statements and related Notes thereto, as well as other cautionary statements and risks described elsewhere in this Annual Report on Form 10-K, before deciding to purchase, hold, or sell shares of our common stock. Overview Our Company and Our Businesses NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. Fueled by th

#### Test search with section filter

In [34]:
# Define a risk-focused query
query = "What supply chain risks does Tesla mention?"

# Search only Tesla Risk Factors section
tesla_risk_results = search_sec_chunks(
    query=query,
    top_k=5,
    ticker="TSLA",
    section_name="item_1a_risk_factors"
)

# Display search results
tesla_risk_results[
    [
        "rank",
        "distance",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,rank,distance,ticker,section_name,citation_label
0,1,0.867302,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
1,2,0.971211,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
2,3,1.006505,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
3,4,1.041513,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
4,5,1.047330,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."


In [35]:
# Create test questions for baseline retrieval
test_queries = [
    {
        "query": "What cybersecurity risks does Microsoft mention?",
        "ticker": "MSFT",
        "section_name": "item_1a_risk_factors"
    },
    {
        "query": "What supply chain risks does Tesla mention?",
        "ticker": "TSLA",
        "section_name": "item_1a_risk_factors"
    },
    {
        "query": "What competition risks does Apple describe?",
        "ticker": "AAPL",
        "section_name": "item_1a_risk_factors"
    },
    {
        "query": "What AI risks does NVIDIA mention?",
        "ticker": "NVDA",
        "section_name": "item_1a_risk_factors"
    }
]

# Create empty list for retrieval test summaries
retrieval_test_records = []

# Loop through each test question
for test in test_queries:

    # Run search
    results_df = search_sec_chunks(
        query=test["query"],
        top_k=3,
        ticker=test["ticker"],
        section_name=test["section_name"]
    )

    # Store top result summary
    retrieval_test_records.append({
        "query": test["query"],
        "ticker_filter": test["ticker"],
        "section_filter": test["section_name"],
        "top_ticker": results_df.iloc[0]["ticker"],
        "top_section": results_df.iloc[0]["section_name"],
        "top_distance": results_df.iloc[0]["distance"],
        "top_citation": results_df.iloc[0]["citation_label"],
        "top_text_preview": results_df.iloc[0]["chunk_text"][:300]
    })

# Convert summaries to DataFrame
retrieval_tests_df = pd.DataFrame(retrieval_test_records)

# Display retrieval tests
retrieval_tests_df

,query,ticker_filter,section_filter,top_ticker,top_section,top_distance,top_citation,top_text_preview
0,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors,MSFT,item_1a_risk_factors,0.861831,"MSFT 2025-07-30 10-K, item_1a_risk_factors, ch...",previously disclosed in our Form 8-K filed wit...
1,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors,TSLA,item_1a_risk_factors,0.867302,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch...",be challenging due to our limited operating hi...
2,What competition risks does Apple describe?,AAPL,item_1a_risk_factors,AAPL,item_1a_risk_factors,0.519304,"AAPL 2025-10-31 10-K, item_1a_risk_factors, ch...",their products to offer more competitive solut...
3,What AI risks does NVIDIA mention?,NVDA,item_1a_risk_factors,NVDA,item_1a_risk_factors,1.016616,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch...","harm, competitive harm or legal liability. Lev..."


#### Save retrieval test results

In [36]:
# Set output path for retrieval test results
retrieval_tests_file = PROCESSED_DIR / "sec_10k_baseline_retrieval_tests.csv"

# Save retrieval test results
retrieval_tests_df.to_csv(retrieval_tests_file, index=False)

# Confirm file was saved
print("Saved retrieval test results to:", retrieval_tests_file)

Saved retrieval test results to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_baseline_retrieval_tests.csv


#### Final checkpoint

In [37]:
# Create final checkpoint table
embedding_checkpoint = pd.DataFrame({
    "output": [
        "Chroma vector database folder",
        "Embedding summary",
        "Baseline retrieval tests"
    ],
    "path": [
        str(CHROMA_DIR),
        str(embedding_summary_file),
        str(retrieval_tests_file)
    ],
    "exists": [
        CHROMA_DIR.exists(),
        embedding_summary_file.exists(),
        retrieval_tests_file.exists()
    ]
})

# Display checkpoint table
embedding_checkpoint

,output,path,exists
0,Chroma vector database folder,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
1,Embedding summary,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
2,Baseline retrieval tests,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True


In [38]:
# Validate that Chroma contains the same number of records as the chunk dataset
collection_count = collection.count()
chunk_count = len(rag_chunks_df)

# Print both counts
print("Rows in chunk dataset:", chunk_count)
print("Records in Chroma collection:", collection_count)

# Confirm the counts match
if collection_count != chunk_count:
    raise ValueError("Chroma collection count does not match chunk dataset count.")

# Confirm retrieval test file exists
print("Retrieval test file exists:", retrieval_tests_file.exists())

# Confirm embedding summary file exists
print("Embedding summary file exists:", embedding_summary_file.exists())

# Final success message
print("Notebook 05 completed successfully.")

Rows in chunk dataset: 520
Records in Chroma collection: 520
Retrieval test file exists: True
Embedding summary file exists: True
Notebook 05 completed successfully.


## Embeddings and Vector Store Conclusion

This notebook converted RAG-ready SEC filing chunks into vector embeddings and stored them in Chroma.

The project now has:

```text
SEC chunks
→ sentence-transformer embeddings
→ Chroma vector database
→ semantic search
→ baseline retrieval tests
```

This is the first working retrieval layer of RiskRadar AI.

The next notebook will focus on baseline retrieval quality:

```text
question
→ retrieve chunks
→ inspect evidence
→ compare results
→ improve retrieval strategy
```

After that, we will add answer generation.